# Riesgo y predicción cuantitativa: factores asociados al mal riesgo crediticio

**Curso:** MCC002 - Probabilidad y Estadística Computacional 

**Grupo 5:** 
- Armando Castro Chaupis 
- Henry Sánchez Alvarado 
- Alex Segura Núñez

**Pregunta principal:** ¿Qué factores explican la probabilidad de que un solicitante sea clasificado como mal riesgo crediticio?

**Preguntas secundarias:**

1. Proporción global de malos créditos y su IC 95 %.
2. ¿La tasa de mal crédito difiere según el propósito del préstamo?
3. ¿La duración y el monto del crédito difieren entre buenos y malos créditos?
4. ¿Qué variables están asociadas con mayor riesgo crediticio?
5. ¿Cómo cambia la clasificación al modificar el umbral de decisión?

## 1. Importación de librerias

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf
import sklearn
from scipy import stats 

### Establecemos valor de semilla
SEED = 2026
np.random.seed(SEED)

### Estilo de gráficos para matplotlib
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12


# 2. Carga de dataset

import os, io, zipfile, urllib.request

UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip"
CANDIDATOS = ["data/german.data", "german.data"]

ruta_datos = next((p for p in CANDIDATOS if os.path.exists(p)), None)

if ruta_datos is None:
    # Descarga documentada desde la fuente oficial (solo si no existe copia local)
    os.makedirs("data", exist_ok=True)
    print("Descargando dataset desde UCI...")
    with urllib.request.urlopen(UCI_ZIP_URL) as r:
        zf = zipfile.ZipFile(io.BytesIO(r.read()))
        zf.extract("german.data", path="data")
    ruta_datos = "data/german.data"

# Nombres de columnas según german.doc (atributos 1..20 + clase)
columnas = [
    "estado_cuenta", "duracion", "historial_credito", "proposito", "monto",
    "ahorros", "empleo_actual", "tasa_cuota", "estatus_personal_sexo",
    "otros_deudores", "anios_residencia", "propiedad", "edad",
    "otros_planes_pago", "vivienda", "n_creditos_banco", "trabajo",
    "n_dependientes", "telefono", "trabajador_extranjero", "clase",
]

df = pd.read_csv(ruta_datos, sep=" ", header=None, names=columnas)
print(f"Archivo cargado: {ruta_datos}")
print(f"Dimensiones: {df.shape[0]} observaciones x {df.shape[1]} columnas")
df.head()

### 3.1 Diccionario de variables

Construido a partir del dataset. Las 13 variables cualitativas usan códigos `A**`. 
Abajo se documenta el significado de cada código y se crea un mapeo de etiquetas.

| # | Variable (nombre en el notebook) | Tipo | Descripción |
|---|---|---|---|
| 1 | `estado_cuenta` | Categórica ordinal | Estado de la cuenta corriente: A11 (< 0 DM), A12 (0–200 DM), A13 (≥ 200 DM), A14 (sin cuenta) |
| 2 | `duracion` | Numérica (meses) | Duración del crédito |
| 3 | `historial_credito` | Categórica | A30–A34: de "sin créditos/todo pagado" a "cuenta crítica/créditos en otros bancos" |
| 4 | `proposito` | Categórica nominal | A40–A410: auto nuevo/usado, mobiliario, radio/TV, electrodomésticos, reparaciones, educación, reentrenamiento, negocio, otros |
| 5 | `monto` | Numérica (DM) | Monto del crédito |
| 6 | `ahorros` | Categórica ordinal | A61–A65: nivel de ahorros/bonos (A65 = desconocido/sin cuenta) |
| 7 | `empleo_actual` | Categórica ordinal | A71–A75: antigüedad en el empleo actual |
| 8 | `tasa_cuota` | Numérica (1–4) | Cuota como % del ingreso disponible |
| 9 | `estatus_personal_sexo` | Categórica | A91–A94: estado civil y sexo (composición histórica del dataset) |
| 10 | `otros_deudores` | Categórica | A101 ninguno, A102 co-solicitante, A103 garante |
| 11 | `anios_residencia` | Numérica (1–4) | Años en la residencia actual |
| 12 | `propiedad` | Categórica | A121 inmueble … A124 sin propiedad conocida |
| 13 | `edad` | Numérica (años) | Edad del solicitante |
| 14 | `otros_planes_pago` | Categórica | A141 banco, A142 tiendas, A143 ninguno |
| 15 | `vivienda` | Categórica | A151 alquilada, A152 propia, A153 gratuita |
| 16 | `n_creditos_banco` | Numérica | N.º de créditos existentes en este banco |
| 17 | `trabajo` | Categórica ordinal | A171–A174: de no calificado/no residente a directivo/independiente |
| 18 | `n_dependientes` | Numérica | Personas a cargo |
| 19 | `telefono` | Binaria | A191 no, A192 sí (registrado) |
| 20 | `trabajador_extranjero` | Binaria | A201 sí, A202 no |
| 21 | `clase` | **Variable respuesta** | 1 = buen riesgo, 2 = mal riesgo |

In [ ]:
# Mapeos de códigos Axy -> etiquetas legibles (según german.doc)
mapa_proposito = {
    "A40": "Auto nuevo", "A41": "Auto usado", "A42": "Mobiliario/equipos",
    "A43": "Radio/TV", "A44": "Electrodomésticos", "A45": "Reparaciones",
    "A46": "Educación", "A48": "Reentrenamiento", "A49": "Negocio", "A410": "Otros",
}
mapa_estado_cuenta = {
    "A11": "< 0 DM", "A12": "0 – 200 DM", "A13": "≥ 200 DM", "A14": "Sin cuenta",
}
mapa_ahorros = {
    "A61": "< 100 DM", "A62": "100 – 500 DM", "A63": "500 – 1000 DM",
    "A64": "≥ 1000 DM", "A65": "Desconocido/sin cuenta",
}
mapa_historial = {
    "A30": "Sin créditos/todo pagado", "A31": "Pagados en este banco",
    "A32": "Al día hasta ahora", "A33": "Retrasos en el pasado",
    "A34": "Cuenta crítica/otros bancos",
}

df["proposito_lbl"] = df["proposito"].map(mapa_proposito)
df["estado_cuenta_lbl"] = df["estado_cuenta"].map(mapa_estado_cuenta)

# Verificación de que no quedaron códigos sin mapear
assert df["proposito_lbl"].notna().all(), "Hay códigos de propósito sin mapear"
assert df["estado_cuenta_lbl"].notna().all(), "Hay códigos de estado de cuenta sin mapear"

vars_numericas = ["duracion", "monto", "tasa_cuota", "anios_residencia",
                  "edad", "n_creditos_banco", "n_dependientes"]
vars_categoricas = [c for c in columnas if c not in vars_numericas + ["clase"]]
print(f"Variables numéricas ({len(vars_numericas)}): {vars_numericas}")
print(f"Variables categóricas ({len(vars_categoricas)}): {vars_categoricas}")

## Parte 2

## 9. Inferencia a través de la frecuencia
### 9.1 ¿La tasa de mal crédito difiere según el propósito?

Deberíamos esperar que para cada propósito exista la misma proporción de créditos de `riesgo` y `no-riesgo` (lo esperado, $E_i$) respecto a la proporción de toda la muestra. Sin embargo, en la práctica esto no siempre ocurre (lo observado, $O_i$). Para evaluar si la tasa de crédito difiere según el propósito, utilizaremos la prueba de **chi-cuadrado** $\chi^2$. Esta prueba se usa debido a la naturaleza categorica de las variables. 

$$
\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}
$$

Recordar que los grados de libertad de la distribucion se calculan $g_{dl}=(r-1)(c-1)$. Donde `r` es el número de propositos y `c` es la cantidad de clases. Para declarar que es válido la prueba de $\chi^2$, se debe cumplir los siguientes puntos:
- No debe existir frecuencias esperadas menor a 1.
- Como máximo, el 20% de las frecuencias esperadas deben ser menores a 5.

Despues de satisfacer esto, esto es posible hallar el valor `p` que nos permite aceptar o rechazar la **hipotesis de independencia** al 5\%. 


En caso se rechace la hipotesis, ¿que tan fuerte es esta dependencia entre variables? Para ello usarémos  **V de Cramér** como tamaño de efecto:
$$V = \sqrt{\frac{\chi^2}{n \cdot \min(r-1, c-1)}}$$

Y se puede interpretar de la siguiente forma:

 | V          | Interpretación    |
| ---------- | ----------------- |
| 0          | Sin relación      |
| 0.10       | Relación pequeña  |
| 0.30       | Relación moderada |
| 0.50 o más | Relación fuerte   |



In [ ]:
## Codigo tal cual

tabla = pd.crosstab(df["proposito_lbl"], df["mal_credito"])
chi2, p_chi, dof, esperadas = stats.chi2_contingency(tabla)
V = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
n_esp_bajas = int((esperadas < 5).sum())

print("--- Prueba chi-cuadrado: clase x propósito (10 categorías) ---")
print(f"chi2 = {chi2:.2f}, gl = {dof}, p = {p_chi:.5f}, V de Cramér = {V:.3f}")
print(f"Celdas con frecuencia esperada < 5: {n_esp_bajas} de {tabla.size}")

In [ ]:
## Permite ver que tipo de variable tiene baja frecuencia esperada.
esp_df = pd.DataFrame(
    esperadas,
    index=tabla.index,
    columns=tabla.columns
)
 
for fila in esp_df.index:
    for col in esp_df.columns:
        if esp_df.loc[fila, col] < 5:
            print(f"{fila} - {col}: {esp_df.loc[fila, col]:.2f}")

A fin de mejorar la robustez de la estimación por la prueba del 

In [ ]:
frec = df["proposito_lbl"].value_counts()
raros = frec[frec < 30].index.tolist()
df["proposito_grp"] = df["proposito_lbl"].where(~df["proposito_lbl"].isin(raros),
                                                "Otros (agrupado)")
tabla_g = pd.crosstab(df["proposito_grp"], df["mal_credito"])
chi2_g, p_g, dof_g, esp_g = stats.chi2_contingency(tabla_g)
V_g = np.sqrt(chi2_g / (n * (min(tabla_g.shape) - 1)))
print(f"\n--- Robustez con categorías raras agrupadas ({tabla_g.shape[0]} categorías) ---")
print(f"Agrupados: {raros}")
print(f"chi2 = {chi2_g:.2f}, gl = {dof_g}, p = {p_g:.5f}, V de Cramér = {V_g:.3f}, "
      f"esperadas < 5: {int((esp_g < 5).sum())}")

De ambos procedimientos, se puede observar que la prueba de Chi-cuadrado muestra dependencia estadistica significativa entre el proposito y el riesgo del credito. Sin embargo, es importante resaltar que el tamaño de efecto `V` es pequeño, indicando que su relación es pequeña.

### 9.2 ¿Difieren la duración y el monto entre buenos y malos créditos?

### 9.2.1 Analisis de la variable duración
Debido a que son variables numericas, realizar una prueba de Chi-cuadrado no serviria. Por lo que, para la categoría **duración**: aplicamos la prueba **t de Welch**. Esta prueba no asume varianzas iguales como la **t de Student**. 

Al igual que en la sección anterior, para estimar que tan fuerte es esta dependencia usamos la `d de Cohen`, definido como:

$$
d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{pooled}}}
$$

donde:

- $\bar{x}_1$: media del grupo 1.
- $\bar{x}_2$: media del grupo 2.
- $s_{\text{pooled}}$: desviación estándar agrupada, calculada como:

$$
s_{\text{pooled}} =
\sqrt{
\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}
{n_1+n_2-2}
}
$$

A fin de interpretar el valor de `d`, se usa la siguiente convección:
- $d \approx 0.2$: efecto pequeño.
- $d \approx 0.5$: efecto mediano.
- $d \ge 0.8$: efecto grande.


In [ ]:
## Selección de buenos y malos creditos
buenos = df[df["mal_credito"] == 0]
malos  = df[df["mal_credito"] == 1]

## Definición de la d cohen

def d_cohen(a, b):
    s_pool = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / s_pool

t_w, p_w = stats.ttest_ind(malos["duracion"], buenos["duracion"], equal_var=False)
d_dur = d_cohen(malos["duracion"], buenos["duracion"])
print("--- Duración (meses): Welch ---")
print(f"media buenos = {buenos['duracion'].mean():.2f} | media malos = {malos['duracion'].mean():.2f}")
print(f"t = {t_w:.3f}, p = {p_w:.2e}, d de Cohen = {d_dur:.3f}")

Con el valor de `p`$<2.4e-10$ se rechaza la hipotesis de igualdad de medias. En la muestra, se observa que en promedio, los créditos malos presentan una duración mayor (24.86 meses) que los créditos buenos (19.21 meses). El tamaño del efecto fue `d`=0.480, correspondiente a un efecto moderado.

### 9.2.2 Analisis de la variable monto 
Para la variable `monto`, se observa que los valores están distribuidos en un rango mucho más amplio que los valores de la variable `duración`. Con el test de `Mann–Whitney` se comparan los rangos, por lo que no es necesario realizar alguna transformación a la variable `monto`.Es común utilizar el tamaño del efecto **$r$**.

Se calcula como:

$$
r = 1 - \frac{2U}{n_1 n_2}
$$

donde:

- $U$: estadístico de la prueba de Mann–Whitney.
- $n_1$: tamaño del primer grupo.
- $n_2$: tamaño del segundo grupo.

El valor de $r$ suele interpretar de la siguiente manera:

| $r$ | Interpretación |
|:-----:|:---------------|
| 0.0 | Sin diferencia |
| 0.1 | Efecto pequeño |
| 0.3 | Efecto moderado |
| 0.5 o mayor | Efecto grande |


Adicionalmente, calculamos que tan fuerte es la dependencia usando el `d de Cohen`. Sin embargo, transformamos los valores a $log(monto)$ debido a la asimetria de los valores `monto`.

In [ ]:
U, p_u = stats.mannwhitneyu(malos["monto"], buenos["monto"], alternative="two-sided")
r_rb = 1 - 2*U/(len(malos)*len(buenos))       
d_logm = d_cohen(malos["log_monto"], buenos["log_monto"])
print("\n--- Monto (DM): Mann-Whitney ---")
print(f"mediana buenos = {buenos['monto'].median():.0f} | mediana malos = {malos['monto'].median():.0f}")
print(f"U = {U:.0f}, p = {p_u:.4f}, r biserial de rangos = {r_rb:.3f}, "
      f"d de Cohen sobre log(monto) = {d_logm:.3f}")

Se observa que los malos creditos tienen montos mayores ($2574$ vs. $2244$) pero el efecto es **pequeño** (d sobre log-monto ≈ 0.24). La dirección negativa de $r$ nos indica el orden del cálculo de las medianas, donde el valor absoluto nos indica una relación pequeña respecto a la calidad del credito.


Sin embargo, es de notar que las variables  `monto` y `duración` están relacionadas por el alto valor de correlación obtenida ($0.62$).

## 10. Regresion lineal 

Para entender la estructura del credito, es decir: ¿que depende o determina el tamaño del monto de un prestamo? Por esta razón realizamos un regresor lineal usando las variables:
$$\log(\text{monto}_i) = \beta_0 + \beta_1\,\text{duracion}_i + \beta_2\,\text{tasa\_cuota}_i + \beta_3\,\text{edad}_i + \varepsilon_i,\qquad \varepsilon_i \sim \mathcal N(0, \sigma^2)$$

In [ ]:
modelo_ols = smf.ols("log_monto ~ duracion + tasa_cuota + edad", data=df).fit()
print(modelo_ols.summary())

### Colinealidad: Factor de Inflación de la Varianza (VIF)

La **multicolinealidad** ocurre cuando una variable explicativa puede predecirse a partir de una combinación de las demás variables del modelo. Cuando esto sucede, los coeficientes estimados pueden volverse inestables y su interpretación resulta más difícil.

Para evaluar este problema se utiliza el **Factor de Inflación de la Varianza (VIF)**, definido como:

$$
VIF_j=\frac{1}{1-R_j^2}
$$

donde:

- $R_j^2$ es el coeficiente de determinación obtenido al ajustar una regresión donde la variable $X_j$ se utiliza como variable respuesta y las demás variables explicativas actúan como predictores.

Si una variable puede explicarse muy bien mediante las demás, entonces $R_j^2$ será cercano a 1 y el VIF crecerá considerablemente, indicando un problema de multicolinealidad.

Como regla general, la interpretación del VIF es la siguiente:

| VIF | Interpretación |
|:---:|:---------------|
| 1 | No existe correlación con las demás variables. |
| 1–2 | Muy baja colinealidad. |
| 2–5 | Colinealidad moderada, generalmente aceptable. |
| 5–10 | Multicolinealidad importante; conviene revisarla. |
| >10 | Multicolinealidad severa; los coeficientes pueden ser inestables y difíciles de interpretar. |

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = sm.add_constant(df[["duracion", "tasa_cuota", "edad"]])
vifs = pd.Series(
    [variance_inflation_factor(X_vif.values, i) for i in range(1, X_vif.shape[1])],
    index=["duracion", "tasa_cuota", "edad"], name="VIF")
print(vifs.round(3))

**Comentario:** Se observa que no hay colinealidad entre variables.

### Analisis de residuos

Al analizar los residuos de un modelo de regresión, se espera que tengan una dispersión aproximadamente constante a lo largo de los valores ajustados.

- **Homocedasticidad:** la varianza de los residuos es constante. En el gráfico de residuos vs. valores ajustados se debe observar una nube de puntos sin un patrón evidente.

- **Heterocedasticidad:** la varianza de los residuos cambia con los valores ajustados (por ejemplo, forma de embudo). Esto puede producir errores estándar sesgados y afectar la validez de las pruebas de hipótesis.

En la practica, la heterocedasticidad puede evaluarse visualmente mediante el gráfico de residuos y confirmarse con pruebas como **Breusch–Pagan**.

In [ ]:

infl = modelo_ols.get_influence()
resid = modelo_ols.resid
fitted = modelo_ols.fittedvalues
resid_std = infl.resid_studentized_internal
leverage = infl.hat_matrix_diag

fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
 
axes[0, 0].scatter(fitted, resid, s=12, alpha=0.5, color="#2c7fb8")
axes[0, 0].axhline(0, color="k", lw=1)

 
sns.regplot(x=fitted, y=resid, lowess=True, scatter=False,
            line_kws={"color": "red", "lw": 1.5}, ax=axes[0, 0])
axes[0, 0].set_xlabel("Valores ajustados"); axes[0, 0].set_ylabel("Residuales")
axes[0, 0].set_title("(1) Residuales vs. ajustados")


 
sm.qqplot(resid, line="45", fit=True, ax=axes[0, 1], markersize=3, alpha=0.5)
axes[0, 1].set_title("(2) QQ-plot de residuales")
 
axes[1, 0].scatter(fitted, np.sqrt(np.abs(resid_std)), s=12, alpha=0.5, color="#2c7fb8")
sns.regplot(x=fitted, y=np.sqrt(np.abs(resid_std)), lowess=True, scatter=False,
            line_kws={"color": "red", "lw": 1.5}, ax=axes[1, 0])
axes[1, 0].set_xlabel("Valores ajustados")
axes[1, 0].set_ylabel(r"$\sqrt{|residual\ estandarizado|}$")
axes[1, 0].set_title("(3) Scale-Location")
 
 
cooks = infl.cooks_distance[0]
sc = axes[1, 1].scatter(leverage, resid_std, s=14, alpha=0.55, c=cooks, cmap="viridis")
axes[1, 1].axhline(0, color="k", lw=1)
axes[1, 1].set_xlabel("Leverage"); axes[1, 1].set_ylabel("Residual estandarizado")
axes[1, 1].set_title("(4) Residuales vs. leverage")
plt.colorbar(sc, ax=axes[1, 1], label="Distancia de Cook")

fig.suptitle("Figura 6. Diagnósticos canónicos del modelo OLS log(monto)", y=1.0)
plt.tight_layout(); plt.show()

print(f"Distancia de Cook máxima: {cooks.max():.4f} (umbral de alarma habitual: 1)")

In [ ]:

infl = modelo_ols.get_influence()
resid = modelo_ols.resid
fitted = modelo_ols.fittedvalues
resid_std = infl.resid_studentized_internal
leverage = infl.hat_matrix_diag

fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))

### Residuales vs. ajustados
axes[0, 0].scatter(fitted, resid, s=12, alpha=0.5, color="#2c7fb8")
axes[0, 0].axhline(0, color="k", lw=1)
 
sns.regplot(x=fitted, y=resid, lowess=True, scatter=False,
            line_kws={"color": "red", "lw": 1.5}, ax=axes[0, 0])
axes[0, 0].set_xlabel("Valores ajustados"); axes[0, 0].set_ylabel("Residuales")
axes[0, 0].set_title("(1) Residuales vs. ajustados")



###QQ-plot de residuales
sm.qqplot(resid, line="45", fit=True, ax=axes[0, 1], markersize=3, alpha=0.5)
axes[0, 1].set_title("(2) QQ-plot de residuales")

### Scale-location
axes[1, 0].scatter(fitted, np.sqrt(np.abs(resid_std)), s=12, alpha=0.5, color="#2c7fb8")
sns.regplot(x=fitted, y=np.sqrt(np.abs(resid_std)), lowess=True, scatter=False,
            line_kws={"color": "red", "lw": 1.5}, ax=axes[1, 0])
axes[1, 0].set_xlabel("Valores ajustados")
axes[1, 0].set_ylabel(r"$\sqrt{|residual\ estandarizado|}$")
axes[1, 0].set_title("(3) Scale-Location")

### Residuales vs. leverage (con distancia de Cook)
cooks = infl.cooks_distance[0]
sc = axes[1, 1].scatter(leverage, resid_std, s=14, alpha=0.55, c=cooks, cmap="viridis")
axes[1, 1].axhline(0, color="k", lw=1)
axes[1, 1].set_xlabel("Leverage"); axes[1, 1].set_ylabel("Residual estandarizado")
axes[1, 1].set_title("(4) Residuales vs. leverage")
plt.colorbar(sc, ax=axes[1, 1], label="Distancia de Cook")

fig.suptitle("Figura 6. Diagnósticos canónicos del modelo OLS log(monto)", y=1.0)
plt.tight_layout(); plt.show()

print(f"Distancia de Cook máxima: {cooks.max():.4f} (umbral de alarma habitual: 1)")

### Test de Shapiro–Wilk

Este test se usa para comprobar si los residuos siguen una distribución aproximadamente normal.

- **Hipótesis nula ($H_0$):** los residuos siguen una distribución normal.
- **Hipótesis alternativa ($H_1$):** los residuos no siguen una distribución normal.

Si el **$p$-valor $> 0.05$**, no se rechaza $H_0$, por lo que la normalidad es una suposición razonable. En caso contrario, existe evidencia de que los residuos no son normales.

Además tambien se revisará el **QQ-Plot** para verificar visualmente la normalidad.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

sw_stat, sw_p = stats.shapiro(resid)
print(f"Shapiro-Wilk : W  = {sw_stat:.4f}, p = {sw_p:.4f}  "
      f"-> {'se rechaza normalidad' if sw_p < 0.05 else 'no se rechaza normalidad'}")


### Test de Breusch–Pagan

Este test se utiliza para verificar si los residuos tienen una varianza constante.

- **Hipótesis nula ($H_0$):** existe homocedasticidad (varianza constante).
- **Hipótesis alternativa ($H_1$):** existe heterocedasticidad (la varianza cambia).

Si el **$p$-valor $> 0.05$**, no se rechaza $H_0$, por lo que la suposición de homocedasticidad es razonable. En caso contrario, existe evidencia de heterocedasticidad.

In [ ]:
bp_stat, bp_p, _, _ = het_breuschpagan(resid, modelo_ols.model.exog)
print(f"Breusch-Pagan: LM = {bp_stat:.2f}, p = {bp_p:.4f}  "
      f"-> {'se rechaza homocedasticidad' if bp_p < 0.05 else 'no se rechaza homocedasticidad'}")

### Errores estándar robustos (HC3)

Los errores estándar robustos (HC3) se utilizan cuando existe heterocedasticidad. No modifican los coeficientes estimados del modelo, pero sí corrigen los errores estándar, los intervalos de confianza y los $p$-valores, haciendo que la inferencia sea más confiable.

In [ ]:
# Errores estándar robustos (HC3) como corrección ante heterocedasticidad
modelo_hc3 = modelo_ols.get_robustcov_results(cov_type="HC3")
tabla_ee = pd.DataFrame({
    "coef": modelo_ols.params,
    "EE clásico": modelo_ols.bse,
    "EE robusto HC3": modelo_hc3.bse,
})
print("\nComparación de errores estándar (clásicos vs. robustos):")
print(tabla_ee.round(4))

### Validación cruzada por k-fold (k=5)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

X_ols = df[["duracion", "tasa_cuota", "edad"]].values
y_ols = df["log_monto"].values
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

resultados_cv = []
for k, (i_tr, i_te) in enumerate(kf.split(X_ols), start=1):
    reg = LinearRegression().fit(X_ols[i_tr], y_ols[i_tr])
    pred = reg.predict(X_ols[i_te])
    resultados_cv.append({
        "fold": k,
        "R2": r2_score(y_ols[i_te], pred),
        "RMSE": np.sqrt(mean_squared_error(y_ols[i_te], pred)),
    })
cv_ols = pd.DataFrame(resultados_cv).set_index("fold")
print(cv_ols.round(4))
print(f"\nR2 medio = {cv_ols['R2'].mean():.4f} ± {cv_ols['R2'].std():.4f} | "
      f"RMSE medio = {cv_ols['RMSE'].mean():.4f} (log-DM)")
print(f"R2 in-sample del modelo completo: {modelo_ols.rsquared:.4f} "
      "(similar al CV: no hay sobreajuste)")

**Conclusión del modelo OLS.** El modelo explica aproximadamente el **54 %** de la variación de $\log(\text{monto})$ ($R^2 \approx 0.54$) y obtiene un resultado muy similar en validación cruzada ($R^2_{\text{CV}} \approx 0.53$), lo que indica que no presenta sobreajuste. La **duración del crédito** es la variable con mayor efecto ($\hat\beta_1 \approx 0.043$), es decir, cada mes adicional se asocia con un incremento cercano al **4.4 %** en el monto esperado, manteniendo constantes las demás variables. En cambio, un mayor porcentaje del ingreso destinado a la cuota se relaciona con montos menores. Además, el **VIF ≈ 1** indica que no existe colinealidad importante entre las variables del modelo.

Respecto a los diagnósticos, los residuos no muestran patrones importantes, el test de **Breusch–Pagan** no encuentra evidencia de heterocedasticidad ($p \approx 0.12$) y ninguna observación presenta una influencia excesiva según la distancia de Cook. Aunque el test de **Shapiro–Wilk** rechaza la normalidad estricta ($p \approx 0.003$), esto es común con muestras grandes ($n = 1000$) y no afecta de forma importante la inferencia. Finalmente, los errores estándar robustos **HC3** son prácticamente iguales a los clásicos, lo que refuerza que las conclusiones del modelo son estables y confiables.

## 11. Regresión logistica

Modelamos  la clasificación de si un prestamo es riesgoso o no:
$$\Pr(Y_i = 1 \mid \mathbf x_i) = \sigma(\mathbf x_i^\top \boldsymbol\beta) = \frac{1}{1 + e^{-\mathbf x_i^\top \boldsymbol\beta}}$$
Cada $e^{\beta_j}$ es un **odds ratio (OR)**: el factor multiplicativo sobre las *odds* de mal crédito por unidad del predictor $j$. 

Para la regresión, usaremos las variables con el respaldo del EDA: estado de la cuenta corriente, duración, log(monto), historial crediticio, ahorros, antigüedad laboral, tasa de cuota, edad, vivienda y otros planes de pago.

**Pasos a seguir:** 
- Partición estratificada 70/30 entrenamiento/prueba (manteniendo constante las proporciones en el dataset de entrenamiento y prueba). 
- Se realizará AUC por validación cruzada estratificada de 5 folds como chequeo de estabilidad, asi como los reportes de metricas al dataset de prueba.

En regresión logística no existe un coeficiente de determinación \(R^2\) equivalente al de la regresión lineal. En su lugar, se utiliza el **Pseudo-\(R^2\) de McFadden**, que mide cuánto mejora el modelo ajustado respecto a un modelo nulo (que solo incluye el intercepto).

Se define como:

$$
R^2_{\text{McFadden}}
=
1
-
\frac{\log L_{\text{modelo}}}
{\log L_{\text{nulo}}}
$$

donde:

- $\log L_{\text{modelo}}$: log-verosimilitud del modelo ajustado.
- $\log L_{\text{nulo}}$: log-verosimilitud del modelo que únicamente incluye el intercepto.


Como referencia general, pueden utilizarse los siguientes criterios:

| Pseudo-$R^2$ | Interpretación |
|:--------------:|:---------------|
| $< 0.10$ | Ajuste débil |
| $0.10 - 0.20$ | Ajuste aceptable |
| $0.20 - 0.40$ | Buen ajuste |
| $> 0.40$ | Ajuste excelente (poco frecuente) |


In [ ]:
### Selección de variables.
from sklearn.model_selection import train_test_split

FORMULA_LOGIT = ("mal_credito ~ C(estado_cuenta) + duracion + log_monto"
                 " + C(historial_credito) + C(ahorros) + C(empleo_actual)"
                 " + tasa_cuota + edad + C(vivienda) + C(otros_planes_pago)")

df_train, df_test = train_test_split(df, test_size=0.30, random_state=SEED,
                                     stratify=df["mal_credito"])
print(f"Entrenamiento: {len(df_train)} obs. (tasa mal crédito = {df_train['mal_credito'].mean():.3f})")
print(f"Prueba       : {len(df_test)} obs. (tasa mal crédito = {df_test['mal_credito'].mean():.3f})")

modelo_logit = smf.logit(FORMULA_LOGIT, data=df_train).fit(disp=0)

### revisar el pseudo-r2
print(f"\nPseudo-R2 de McFadden (train): {modelo_logit.prsquared:.4f}")
print(f"Convergencia: {modelo_logit.mle_retvals['converged']}")

Se obtiene un Pseudo-$R^2$ de $0.2202$, lo que nos explica que es un buen ajuste para nuestro clasificador logistico.

### Interpretación del valor \(p\) en la regresión logística

En una regresión logística, cada coeficiente se evalúa mediante una **prueba de hipótesis** para determinar si la variable está asociada significativamente con la probabilidad del evento de interés.

Las hipótesis son:

$$
H_0:\ \beta_j = 0
$$

$$
H_1:\ \beta_j \neq 0
$$

donde $\beta_j$ es el coeficiente asociado a la variable $j$.

El **valor $p$** representa la probabilidad de observar un coeficiente tan extremo como el estimado suponiendo que la hipótesis nula sea verdadera

La decisión se basa en un nivel de significancia, generalmente $\alpha = 0.05$:

- Si $p < 0.05$, se **rechaza la hipótesis nula**, concluyendo que la variable presenta una asociación estadísticamente significativa con la respuesta.
- Si $p \ge 0.05$, **no se rechaza la hipótesis nula**, por lo que no existe evidencia suficiente para afirmar que la variable tenga un efecto sobre la respuesta.


El valor $p$ **no mide el tamaño del efecto**. Una variable puede ser estadísticamente significativa ($p < 0.05$) pero tener un efecto pequeño. Por ello, el valor $p$ debe interpretarse conjuntamente con el **Odds Ratio (OR)** y su **intervalo de confianza del 95%**. Donde la interpretación de este ultimo es la siguiente:

- Si el **IC del 95% no contiene el valor 1**, existe evidencia de que la asociación entre la variable y la respuesta es estadísticamente significativa al nivel del 5%.
- Si el **IC del 95% contiene el valor 1**, no existe evidencia suficiente para afirmar que la variable tenga un efecto significativo sobre la respuesta.

In [ ]:
or_tabla = pd.DataFrame({
    "OR": np.exp(modelo_logit.params),
    "IC 2.5%": np.exp(modelo_logit.conf_int()[0]),
    "IC 97.5%": np.exp(modelo_logit.conf_int()[1]),
    "p-valor": modelo_logit.pvalues,
}).drop(index="Intercept").sort_values("OR")
or_tabla.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
tab = or_tabla.sort_values("OR")
colores = ["#2c7fb8" if o < 1 else "#d95f02" for o in tab["OR"]]
ax.errorbar(tab["OR"], range(len(tab)),
            xerr=[tab["OR"] - tab["IC 2.5%"], tab["IC 97.5%"] - tab["OR"]],
            fmt="o", ecolor="gray", elinewidth=1.2, capsize=3,
            markerfacecolor="white", markeredgecolor="k", zorder=3)
ax.scatter(tab["OR"], range(len(tab)), c=colores, s=42, zorder=4)
ax.axvline(1, color="k", ls="--", lw=1)
ax.set_yticks(range(len(tab))); ax.set_yticklabels(tab.index, fontsize=8.5)
ax.set_xscale("log")
ax.set_xlabel("Odds ratio (escala log) — OR > 1 aumenta el riesgo")
ax.set_title("Figura 7. Odds ratios del modelo logístico (IC 95 %)")
plt.tight_layout(); plt.show()

### Respuesta a la pregunta 4: 

Por ejemplo, para el estado de la cuenta corriente, la categoría **A14** (sin cuenta corriente) presenta un **OR ≈ 0.18** respecto a la categoría de  `saldo negativo`. Esto significa que, manteniendo constantes las demás variables, las *odds* de presentar un mal crédito son aproximadamente un **82% menores** que las del grupo de referencia. De forma similar, la categoría **A13** (saldo mayor o igual a 200 DM) presenta un **OR ≈ 0.31**, por lo que también se asocia con una reducción importante de las *odds* de `mal crédito`.

Algo parecido ocurre con los ahorros, donde los clientes con **ahorros mayores o iguales a 1000 DM (A64)** presentan un **OR ≈ 0.16**, mientras que aquellos cuyo nivel de ahorro es **desconocido (A65)** presentan un **OR ≈ 0.43**. En ambos casos las *odds* de mal crédito son menores que las de la categoría de referencia. También se obtiene un OR menor que 1 para los clientes con **vivienda propia (A152)** y para quienes tienen una **antigüedad laboral entre 4 y 7 años (A74)**.

En cambio, la **duración del crédito** presenta un **OR ≈ 1.04 por cada mes adicional**. Como este valor es mayor que 1, cada mes extra incrementa las *odds* de mal crédito aproximadamente un **4%**, manteniendo constantes las demás variables. Debido a que este efecto es multiplicativo, un año adicional de duración equivale aproximadamente a un incremento del **60%** en las *odds*.. De manera similar, una mayor **tasa de cuota** también incrementa las *odds* de mal crédito (OR ≈ 1.23).

Un resultado interesante es el del **monto del crédito**. En el análisis bivariado (Sección 9.2) se encontró que los créditos malos tenían montos mayores que los créditos buenos. Sin embargo, en la regresión logística el **logaritmo del monto** presenta un **OR ≈ 1.07** con un **valor $p \approx 0.72$**, por lo que deja de ser estadísticamente significativo. Esto indica que, una vez consideradas las demás variables del modelo, el monto ya no aporta información adicional para explicar el riesgo de mal crédito.

La razón es que el monto y la duración están correlacionados (\(\rho \approx 0.62\)). En general, los créditos de mayor monto también suelen tener plazos más largos. Como la duración ya está incluida en el modelo y explica una parte importante del riesgo, el efecto que inicialmente parecía atribuirse al monto queda explicado por dicha variable. 

Finalmente, el resultado obtenido para la categoría **A34** debido a que presenta un **OR ≈ 0.24**, lo que sugiere menores *odds* de mal crédito respecto a la categoría de referencia **A30**. 

### 11.1 Desempeño del clasificador: 

Se usarán las metricas ROC–AUC, matriz de confusión, precision y recall para evaluación del modelo.

$$
Recall = \frac{TP}{TP+FN}
$$
$$
Precision = \frac{TP}{TP+FP}
$$

`Falso negativo`=> Cliente malo aprobado.

`Falso positivo`=> Cliente bueno rechazado.

`ROC–AUC`: probabilidad de que el modelo asigne mayor score a un cliente malo que a uno bueno elegidos al azar; es independiente del umbral y del balance de clases.

In [ ]:
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             precision_score, recall_score, f1_score, accuracy_score)

p_test = modelo_logit.predict(df_test)          # probabilidades predichas en prueba
y_test = df_test["mal_credito"].values

auc_test = roc_auc_score(y_test, p_test)
fpr, tpr, umbrales_roc = roc_curve(y_test, p_test)



print(f"ROC-AUC (test, n = {len(df_test)}): {auc_test:.4f}")


**Comentario:** El valor de AUC obtenido es consistente, por lo que se puede decir que el modelo generaliza y no está sobreajustado. 

### Estadístico KS (Kolmogorov–Smirnov)

El estadístico **KS** es muy utilizado en credit scoring. Mide la máxima diferencia entre las CDF de los scores para `riesgo` y `no riesgo`:

$$ KS = \max_x | F_{bad}(x) - F_{good}(x) | $$

Valores típicos:
- KS \< 0.2 → modelo débil.
- KS ~ 0.2–0.4 → modelo razonable.
- KS \> 0.4 → modelo fuerte (regla muy general, depende del negocio).

In [ ]:
# Estadístico KS (máxima separación entre TPR y FPR, usual en scoring crediticio)
ks = float(np.max(tpr - fpr))
umbral_ks = float(umbrales_roc[np.argmax(tpr - fpr)])
print(f"Estadístico KS: {ks:.4f} (alcanzado en umbral = {umbral_ks:.3f})")

**Comentario:** Se obtiene del modelo un valor KS de 0.481, explicando esto que el modelo es un modelo fuerte. 

In [ ]:
# Validación cruzada por 5 folds 
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

cat_logit = ["estado_cuenta", "historial_credito", "ahorros", "empleo_actual",
             "vivienda", "otros_planes_pago"]
num_logit = ["duracion", "log_monto", "tasa_cuota", "edad"]

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", OneHotEncoder(drop="first"), cat_logit),
        ("num", StandardScaler(), num_logit),
    ])),
    ("clf", LogisticRegression(max_iter=2000, C=1e6)),   # C alto ~ sin regularización (comparable a MLE)
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
aucs_cv = cross_val_score(pipe, df[cat_logit + num_logit], df["mal_credito"],
                          cv=skf, scoring="roc_auc")
print("AUC por fold:", np.round(aucs_cv, 4))
print(f"AUC CV 5-fold: {aucs_cv.mean():.4f} ± {aucs_cv.std():.4f}")

**Comentario:** El AUC anterior (~0.77) es consistente con el AUC de validación cruzada (~0.77 ± 0.02): el modelo generaliza y no está sobreajustado

## 12. Análisis del umbral de decisión con costo asimétrico

La matriz de costos para el dataset se plantea de la siguiente forma. De forma aleatoria fue planteado que:

| Real \ Predicho | Bueno | Malo |
|---|---|---|
| **Bueno** | 0 | 1 (FP) |
| **Malo** | **5 (FN)** | 0 |

La teoría de decisión bayesiana da el umbral óptimo que minimiza el costo esperado: clasificar como *malo* cuando el umbral sea:
$$
t^{*}
=
\frac{c_{FP}}{c_{FP}+c_{FN}}
=
\frac{1}{1+5}
=
\frac{1}{6}
\approx 0.167.
$$

Comparamos el umbral estándar $t = 0.50$ con $t^{*} = 1/6$ y trazamos la curva de costo empírico total en prueba para todo $t$ a fin de evaluar los costos minimos.

In [ ]:
C_FN, C_FP = 5, 1    

def evaluar_umbral(t, y, p):
    yhat = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return {
        "umbral": t, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "accuracy": accuracy_score(y, yhat),
        "precision": precision_score(y, yhat, zero_division=0),
        "recall": recall_score(y, yhat),
        "F1": f1_score(y, yhat),
        "costo_total": C_FN * fn + C_FP * fp,
        "costo_medio": (C_FN * fn + C_FP * fp) / len(y),
    }

t_opt_teorico = C_FP / (C_FP + C_FN)
comparacion = pd.DataFrame([
    evaluar_umbral(0.50, y_test, p_test),
    evaluar_umbral(t_opt_teorico, y_test, p_test),
]).set_index("umbral").round(4)
print(f"Umbral óptimo teórico bajo costo 5:1 -> t* = 1/6 = {t_opt_teorico:.4f}\n")
comparacion

In [ ]:
# Curva de costo empírico vs. umbral (prueba) y umbral empírico de costo mínimo
rejilla_t = np.linspace(0.01, 0.99, 197)
costos = np.array([evaluar_umbral(t, y_test, p_test)["costo_total"] for t in rejilla_t])
t_emp = float(rejilla_t[np.argmin(costos)])

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(rejilla_t, costos, color="#2c7fb8", lw=2)
ax.axvline(0.50, color="gray", ls=":", lw=1.5, label="t = 0.50 (estándar)")
ax.axvline(t_opt_teorico, color="#d95f02", ls="--", lw=1.8,
           label=f"t* = 1/6 ≈ {t_opt_teorico:.3f} (óptimo teórico 5:1)")
ax.scatter([t_emp], [costos.min()], color="k", zorder=5,
           label=f"mínimo empírico t = {t_emp:.2f} (costo = {int(costos.min())})")
ax.set_xlabel("Umbral de decisión t"); ax.set_ylabel("Costo total en prueba (5·FN + 1·FP)")
ax.set_title("Figura 8. Costo esperado vs. umbral de decisión")
ax.legend()
plt.tight_layout(); plt.show()

**Respuesta a la pregunta 5:** 

Al bajar el umbral de 0.50 a $t^{*} = 1/6$:

- El **recall** de malos créditos casi se duplica (de 0.47 a 0.89): los falsos negativos caen de 48 a 10.

- La **precision** cae (de 0.58 a 0.43) y el **accuracy** baja (de 0.74 a 0.61) — más buenos clientes rechazados (FP: de 30 a 108).

- Pero el **costo total** en prueba cae ≈ 41 % (de 270 a 158), porque cada FN evitado cuesta hasta 5 FP nuevos.

El mínimo empírico de la curva de costo está cerca del óptimo teórico. La elección del umbral en este caso es una **decisión económica** que debe fijarse con la matriz de costos dependendiendo del negocio.